In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce.silver")

In [0]:
orders = spark.read.table("ecommerce.bronze.orders")
customers = spark.read.table("ecommerce.bronze.customers")
items = spark.read.table("ecommerce.bronze.order_items")
products = spark.read.table("ecommerce.bronze.products")
payments = spark.read.table("ecommerce.bronze.order_payments")
reviews = spark.read.table("ecommerce.bronze.order_reviews")
sellers = spark.read.table("ecommerce.bronze.sellers")
geolocation = spark.read.table("ecommerce.bronze.geolocation")

In [0]:
def standardize_col_name(col_name):
    return col_name.strip().replace(' ', '_').lower()

def rename_columns(df):
    return df.toDF(*[standardize_col_name(c) for c in df.columns])

orders = rename_columns(orders)
customers = rename_columns(customers)
items = rename_columns(items)
products = rename_columns(products)
payments = rename_columns(payments)
reviews = rename_columns(reviews)
sellers = rename_columns(sellers)
geolocation = rename_columns(geolocation)

In [0]:
products = products.withColumnRenamed(
    "product_name_lenght", "product_name_length"
)
products = products.withColumnRenamed(
    "product_description_lenght", "product_description_length"
)

In [0]:
items = items.dropDuplicates(["order_id", "order_item_id"])
reviews = reviews.dropDuplicates(["review_id"])

In [0]:
customers = customers.fillna({
    "customer_city": "unknown",
    "customer_state": "unknown"
})

products = products.fillna({
    "product_category_name": "unknown"
})

In [0]:
orders = orders \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date"))

items = items \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

payments = payments \
    .withColumn("payment_value", col("payment_value").cast("double"))

In [0]:
payments_agg = payments.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_payment"))

In [0]:
orders = orders.withColumn(
    "is_late_delivery",
    col("order_delivered_customer_date") > col("order_estimated_delivery_date")
)

orders = orders.withColumn(
    "invalid_delivery",
    col("order_delivered_customer_date") < col("order_purchase_timestamp")
)


In [0]:
orders = orders.join(customers, "customer_id", "inner")

In [0]:
orders.write.mode("overwrite").saveAsTable("ecommerce.silver.orders")
customers.write.mode("overwrite").saveAsTable("ecommerce.silver.customers")
items.write.mode("overwrite").saveAsTable("ecommerce.silver.order_items")
products.write.mode("overwrite").saveAsTable("ecommerce.silver.products")
payments_agg.write.mode("overwrite").saveAsTable("ecommerce.silver.payments")
reviews.write.mode("overwrite").saveAsTable("ecommerce.silver.reviews")
sellers.write.mode("overwrite").saveAsTable("ecommerce.silver.sellers")
geolocation.write.mode("overwrite").saveAsTable("ecommerce.silver.geolocation")